In [ ]:
import os, re, glob, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

PRE = {
    "ROOT"        : "/kaggle/input/datasets/esmeabha/isleep/Dataset",
    "OUT"         : "/kaggle/working/isleep_cache",
    "TARGET_FS"   : 100,          # paper downsamples EEG/EOG 128 -> 100 Hz
    "EPOCH_SEC"   : 30,
    "BANDPASS"    : (0.3, 35.0),
    "MAX_SUBJECTS": None,         # <-- set to 10 for a quick trial run
}
os.makedirs(PRE["OUT"], exist_ok=True)

# -----------------------------------------------------------------------------
# Channel canonicalisation.
# Batch-1 uses E1:M2/C4:M1..., Batch-4/5 uses EOG1:A2/C4:A1..., and F4/F3 are
# missing from much of Batch-1 while EMG/ECG are missing from Batch-2 onward.
# These six are the intersection across all 97 recordings.
# -----------------------------------------------------------------------------
CANON = {"EOG_L": ("E1", "EOG1"), "EOG_R": ("E2", "EOG2"),
         "C4": ("C4",), "C3": ("C3",), "O2": ("O2",), "O1": ("O1",)}
KEEP = list(CANON)


def canon_picks(ch_names):
    """Map this file's channel names onto the six canonical ones."""
    pre = {c: re.split(r"[:\-]", c)[0].strip().upper() for c in ch_names}
    picks = {}
    for canon, aliases in CANON.items():
        for c, p in pre.items():
            if p in aliases:
                picks[canon] = c
                break
    return picks


# -----------------------------------------------------------------------------
# Hypnogram parsing.  AASM 5-class: N4 folds into N3, Movement is dropped.
# 'A' is this exporter's code for Wake -- confirmed against the Light sheet,
# which spells out 'Wake' at the same timestamps.
# -----------------------------------------------------------------------------
STAGE_MAP = {
    "W": "W", "WAKE": "W", "WACH": "W",
    "N1": "N1", "S1": "N1", "1": "N1",
    "N2": "N2", "S2": "N2", "2": "N2",
    "N3": "N3", "S3": "N3", "3": "N3",
    "N4": "N3", "S4": "N3", "4": "N3",          # AASM folds N4 into N3
    "R": "REM", "REM": "REM",
}
CLASSES = ["W", "N1", "N2", "N3", "REM"]
# The paper documents 'Sleep stage ?' and 'Artifact' for unscorable epochs
# (typically after Lights On). 'Others' is 2.08% of all annotations.
DROP = {"?", "ARTIFACT", "ARTEFACT", "OTHERS", "OTHER", "MOVEMENT", "MT",
        "BODY EVENT", "NAN", "NONE", ""}
# 'A' is genuinely ambiguous in these exports -- it appears where the Light
# sheet spells out 'Wake', but 'Artifact' is also plausible. Resolved per
# subject against a spelled-out Sleep stage column; see resolve_ambiguous().
AMBIGUOUS = {"A"}


def norm_stage(tok):
    """'Sleep stage N2' -> 'N2';  'Wake' -> 'WAKE';  strips the AASM prefix."""
    t = str(tok).strip().upper()
    t = re.sub(r"^SLEEP\s*STAGE\s*", "", t)
    return t.strip()


STAGE_SHEETS = ["Light", "Position", "Pos.", "SpO2", "Heart Rate"]


def read_stage_column(xl):
    """Several sheets carry a spelled-out 'Sleep stage' column alongside their
    own signal. Used to disambiguate single-letter codes in Sleep profile."""
    for name in STAGE_SHEETS:
        sheet = next((s for s in xl.sheet_names if s.strip().lower() == name.lower()), None)
        if sheet is None:
            continue
        try:
            d = xl.parse(sheet, header=None)
        except Exception:
            continue
        for i in range(min(20, len(d))):
            row = [str(v).strip().lower() for v in d.iloc[i].tolist()]
            if "sleep stage" in row and "time" in row:
                tcol, scol = row.index("time"), row.index("sleep stage")
                body = d.iloc[i + 1:, :]
                times = pd.to_datetime(body.iloc[:, tcol], errors="coerce", dayfirst=True)
                ok = times.notna()
                return dict(zip(times[ok], body.iloc[:, scol][ok].astype(str)))
    return None


def read_hypnogram(xlsx):
    """Returns (start_time, list_of_stage_strings) from the Sleep profile sheet."""
    try:
        xl = pd.ExcelFile(xlsx)
    except Exception as e:
        return None, None, f"cannot open xlsx ({e})"
    sheet = next((s for s in xl.sheet_names if s.strip().lower() == "sleep profile"), None)
    if sheet is None:
        return None, None, "no 'Sleep profile' sheet"

    d = xl.parse(sheet, header=None)
    col0, col1 = d.iloc[:, 0], d.iloc[:, 1]

    start = None
    for i in range(min(12, len(d))):
        if str(col0.iloc[i]).strip().lower() == "start time":
            start = pd.to_datetime(col1.iloc[i], errors="coerce", dayfirst=True)
    hdr = next((i for i in range(min(20, len(d)))
                if str(col0.iloc[i]).strip().lower() == "time"), None)
    if hdr is None:
        return None, None, "no Time/Value header row"

    body = d.iloc[hdr + 1:, :2].dropna(how="all")
    times = pd.to_datetime(body.iloc[:, 0], errors="coerce", dayfirst=True)
    stages = body.iloc[:, 1].astype(str).str.strip()
    ok = times.notna()
    if start is None and ok.any():
        start = times[ok].iloc[0]
    rows = list(zip(times[ok], stages[ok]))
    rows = resolve_ambiguous(rows, read_stage_column(xl))
    return start, rows, None


def resolve_ambiguous(rows, alt):
    """Replace single-letter codes with the spelled-out stage from another
    sheet at the same timestamp. Leaves the token alone if nothing matches,
    in which case it surfaces in unmapped_stage_tokens rather than being
    silently guessed at."""
    if not alt:
        return rows
    out = []
    for t, sgl in rows:
        out.append((t, alt[t] if norm_stage(sgl) in AMBIGUOUS and t in alt else sgl))
    return out


def stages_to_labels(rows, start, n_epochs, epoch_sec):
    """Place each scored stage at its epoch index; -1 means unscored/dropped."""
    lab = np.full(n_epochs, -1, dtype=np.int8)
    unmapped = {}
    for t, s in rows:
        i = int(round((t - start).total_seconds() / epoch_sec))
        if not (0 <= i < n_epochs):
            continue
        key = norm_stage(s)
        if key in DROP:
            continue
        m = STAGE_MAP.get(key)
        if m is None:
            unmapped[key] = unmapped.get(key, 0) + 1
            continue
        lab[i] = CLASSES.index(m)
    return lab, unmapped


# -----------------------------------------------------------------------------
# Signal side
# -----------------------------------------------------------------------------
def process_subject(edf, xlsx):
    import mne
    mne.set_log_level("ERROR")

    start, rows, err = read_hypnogram(xlsx)
    if err:
        return None, err

    raw = mne.io.read_raw_edf(edf, preload=False, verbose=False)
    picks = canon_picks(raw.ch_names)
    missing = [c for c in KEEP if c not in picks]
    if missing:
        return None, f"missing channels {missing}"

    raw.pick([picks[c] for c in KEEP])
    raw.load_data()
    raw.reorder_channels([picks[c] for c in KEEP])
    raw.filter(*PRE["BANDPASS"], verbose=False)
    raw.resample(PRE["TARGET_FS"], verbose=False)

    fs = PRE["TARGET_FS"]
    L = fs * PRE["EPOCH_SEC"]
    data = raw.get_data()                                   # (6, T)
    n = data.shape[1] // L
    if n < 10:
        return None, f"only {n} epochs"
    ep = data[:, :n * L].reshape(len(KEEP), n, L).transpose(1, 0, 2)

    lab, unmapped = stages_to_labels(rows, start, n, PRE["EPOCH_SEC"])
    keep = lab >= 0
    if keep.sum() < 10:
        return None, f"only {keep.sum()} scored epochs (check time alignment)"

    ep = ep[keep].astype(np.float32)
    # per-epoch, per-channel z-score -- cannot leak between train and test
    ep = (ep - ep.mean(-1, keepdims=True)) / (ep.std(-1, keepdims=True) + 1e-8)
    ep = np.clip(ep, -20, 20).astype(np.float16)            # halves the cache size
    return (ep, lab[keep].astype(np.int8), unmapped), None


def build_cache():
    edfs = sorted(glob.glob(os.path.join(PRE["ROOT"], "**", "SN*.edf"), recursive=True),
                  key=lambda p: int(re.search(r"SN(\d+)", os.path.basename(p)).group(1)))
    if PRE["MAX_SUBJECTS"]:
        edfs = edfs[:PRE["MAX_SUBJECTS"]]
    print(f"{len(edfs)} EDF files found\n")

    Xs, ys, gs, manifest, all_unmapped = [], [], [], [], {}
    for k, edf in enumerate(edfs, 1):
        sid = os.path.splitext(os.path.basename(edf))[0]
        xlsx = os.path.join(os.path.dirname(edf), sid + ".xlsx")
        if not os.path.exists(xlsx):
            print(f"[{k:3d}/{len(edfs)}] {sid:6s} SKIP  no xlsx"); continue
        try:
            out, err = process_subject(edf, xlsx)
        except Exception as e:
            out, err = None, f"{type(e).__name__}: {e}"
        if out is None:
            print(f"[{k:3d}/{len(edfs)}] {sid:6s} SKIP  {err}"); continue

        ep, lab, unmapped = out
        for u, c in unmapped.items():
            all_unmapped[u] = all_unmapped.get(u, 0) + c
        Xs.append(ep); ys.append(lab)
        gs.append(np.full(len(lab), int(re.search(r"\d+", sid).group()), dtype=np.int16))
        manifest.append({"subject": sid, "epochs": int(len(lab))})
        dist = {CLASSES[i]: int((lab == i).sum()) for i in range(5)}
        print(f"[{k:3d}/{len(edfs)}] {sid:6s} {len(lab):5d} epochs  {dist}")

    if not Xs:
        raise RuntimeError("Nothing processed. Check PRE['ROOT'] and the skip reasons above.")

    X = np.concatenate(Xs); y = np.concatenate(ys); g = np.concatenate(gs)
    del Xs
    np.save(f"{PRE['OUT']}/X.npy", X)
    np.save(f"{PRE['OUT']}/y.npy", y)
    np.save(f"{PRE['OUT']}/groups.npy", g)
    json.dump({"classes": CLASSES, "channels": KEEP, "fs": PRE["TARGET_FS"],
               "epoch_sec": PRE["EPOCH_SEC"], "subjects": manifest,
               "unmapped_stage_tokens": all_unmapped},
              open(f"{PRE['OUT']}/meta.json", "w"), indent=2)

    print(f"\ncache written to {PRE['OUT']}")
    print(f"X {X.shape} {X.dtype}  ({X.nbytes/1e9:.2f} GB)   subjects: {len(manifest)}")
    for i, c in enumerate(CLASSES):
        n = int((y == i).sum())
        print(f"  {c:4s} {n:7d}  ({100*n/len(y):5.1f}%)")
    if all_unmapped:
        print("\n!! unmapped stage tokens (add them to STAGE_MAP):", all_unmapped)


build_cache()

97 EDF files found

[  1/97] SN1      717 epochs  {'W': 254, 'N1': 72, 'N2': 364, 'N3': 25, 'REM': 2}
[  2/97] SN3      973 epochs  {'W': 156, 'N1': 357, 'N2': 384, 'N3': 22, 'REM': 54}
[  3/97] SN4     1025 epochs  {'W': 122, 'N1': 105, 'N2': 446, 'N3': 177, 'REM': 175}
[  4/97] SN5      829 epochs  {'W': 59, 'N1': 87, 'N2': 421, 'N3': 154, 'REM': 108}
[  5/97] SN6      948 epochs  {'W': 328, 'N1': 68, 'N2': 341, 'N3': 108, 'REM': 103}
[  6/97] SN7      947 epochs  {'W': 150, 'N1': 50, 'N2': 479, 'N3': 234, 'REM': 34}
